# 03 · Evaluar en test y exportar a TFLite

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Salvadrn/isic-pi/blob/main/notebooks/03_exportar_tflite.ipynb)

Sesión 3 del proyecto **isic-pi**. Toma el modelo que entrenó el notebook 02, lo mide **por primera vez en el conjunto de test** y lo convierte al formato que corre en la Raspberry Pi.

**Qué hace este notebook**
1. Carga el test (que ningún notebook anterior tocó) y el modelo de `Mi unidad/isic-pi/modelo/`.
2. Mide el modelo en test: accuracy, recall por clase y su promedio, F1 y matriz de confusión.
3. Convierte a TFLite: `modelo.tflite` (float32, idéntico a Keras) y `modelo_ligero.tflite` (pesos en int8, unas 4 veces más chico).
4. Vuelve a medir los dos `.tflite` en test con **el mismo intérprete que usará la Pi** (`ai-edge-litert`) y comprueba que el float32 da lo mismo que Keras.
5. Guarda todo en `Mi unidad/isic-pi/exportado/` y descarga un `.zip` para tu Mac, con unas fotos de muestra para comprobar en la Pi que da los mismos resultados.

**Entorno de ejecución:** CPU basta (la GPU no acelera el intérprete de TFLite).

**Requisito:** haber corrido los notebooks 01 y 02.

> Primera vez: pon `MODO_PRUEBA = True` (usa `datos_prueba/` y `modelo_prueba/` de los notebooks 01 y 02 en modo prueba) para comprobar que todo corre.

## 1. Configuración

In [ ]:
MODO_PRUEBA = False          # True: usa datos_prueba/ y modelo_prueba/ (notebooks 01 y 02 en modo prueba)
FORZAR = False               # True: rehacer aunque ya exista un modelo exportado
USAR_DRIVE = True
CARPETA_DRIVE = "isic-pi"

EXPORTAR_LIGERO = True       # además del float32, una versión con pesos en int8 (~4 veces más chica)
MUESTRAS_POR_CLASE_PI = 2    # fotos de test que viajan con el modelo para comprobar en la Pi que da lo mismo
HILOS = 2                    # hilos del intérprete al medir aquí (Colab tiene 2 núcleos)
SEMILLA = 42

## 2. Instalar el intérprete de la Pi

In [ ]:
%pip install -q ai-edge-litert

In [ ]:
import json, os, shutil, sys, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras

try:
    from ai_edge_litert.interpreter import Interpreter
    RUNTIME = "ai_edge_litert"
except ImportError:
    Interpreter = tf.lite.Interpreter
    RUNTIME = "tf.lite (ai_edge_litert no se pudo importar)"
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| intérprete:", RUNTIME)

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB and USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive") / CARPETA_DRIVE
elif EN_COLAB:
    BASE = Path("/content") / CARPETA_DRIVE
else:
    BASE = Path.cwd() / CARPETA_DRIVE              # fuera de Colab (pruebas locales)

DATOS = BASE / ("datos_prueba" if MODO_PRUEBA else "datos")              # notebook 01
MODELO = BASE / ("modelo_prueba" if MODO_PRUEBA else "modelo")           # notebook 02
SALIDA = BASE / ("exportado_prueba" if MODO_PRUEBA else "exportado")     # este notebook
TRABAJO = (Path("/content") if EN_COLAB else Path.cwd()) / "trabajo" / SALIDA.name

for marca, nb in [(DATOS / "meta.json", "01"), (MODELO / "entrenamiento.json", "02")]:
    if not marca.exists():
        raise FileNotFoundError(f"No encuentro {marca}. Corre primero el notebook {nb} (con el mismo MODO_PRUEBA).")
if (SALIDA / "metricas_test.json").exists() and not FORZAR:
    raise RuntimeError(f"Ya hay un modelo exportado en {SALIDA}. Pon FORZAR = True para rehacerlo.")
SALIDA.mkdir(parents=True, exist_ok=True)
TRABAJO.mkdir(parents=True, exist_ok=True)

meta = json.loads((DATOS / "meta.json").read_text())
info = json.loads((MODELO / "entrenamiento.json").read_text())
CLASES, TAMANO = meta["clases"], meta["tamano"]
K = len(CLASES)
assert info["clases"] == CLASES, "las clases del modelo no coinciden con las de los datos"
assert (MODELO / "clases.txt").read_text().split() == CLASES
print(f"Clases ({K}): {CLASES} | modelo entrenado el {info['fecha']} | fase elegida: {info['fase_elegida']}")

## 3. Cargar el test y el modelo

In [ ]:
def traer(nombre):
    origen, destino = DATOS / nombre, TRABAJO / nombre
    if origen.resolve() == destino.resolve():
        return destino
    if destino.exists() and destino.stat().st_size == origen.stat().st_size:
        return destino
    t0 = time.time()
    shutil.copy2(origen, destino)
    print(f"{nombre}: {destino.stat().st_size / 1e9:.2f} GB copiados en {time.time() - t0:.0f} s")
    return destino


for nombre in ["test_x.npy", "test_y.npy"]:
    traer(nombre)

X_test = np.load(TRABAJO / "test_x.npy", mmap_mode="r")
y_test = np.load(TRABAJO / "test_y.npy").astype(np.int64)
assert X_test.shape == (len(y_test), TAMANO, TAMANO, 3)
modelo = keras.models.load_model(str(MODELO / "modelo.keras"), compile=False)
print(f"Test: {len(y_test):,} imágenes | parámetros del modelo: {modelo.count_params():,}")

## 4. Medición honesta en test (modelo Keras)

Es la primera vez que el modelo ve estas fotos: no se usaron para entrenar ni para elegir nada. Esta es la cifra que va al reporte.

Si el recall promedio en test sale bastante más bajo que en validación (notebook 02), el modelo se ajustó demasiado a las fotos de entrenamiento.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score, recall_score


def predecir_keras(X, lote=128):
    partes = []
    for ini in range(0, len(X), lote):
        partes.append(modelo.predict(np.asarray(X[ini:ini + lote], dtype=np.float32), verbose=0))
    return np.concatenate(partes)


def metricas(prob):
    pred = prob.argmax(axis=1)
    por_clase = recall_score(y_test, pred, labels=list(range(K)), average=None, zero_division=0)
    return {
        "accuracy": float((pred == y_test).mean()),
        "recall_promedio": float(por_clase.mean()),
        "f1_macro": float(f1_score(y_test, pred, labels=list(range(K)), average="macro", zero_division=0)),
        "recall_por_clase": {c: float(r) for c, r in zip(CLASES, por_clase)},
    }


t0 = time.perf_counter()
prob_keras = predecir_keras(X_test)
ms_keras = (time.perf_counter() - t0) / len(y_test) * 1000
m_keras = metricas(prob_keras)

reporte = classification_report(y_test, prob_keras.argmax(axis=1), labels=list(range(K)),
                                target_names=CLASES, digits=3, zero_division=0)
print(reporte)
print(f"Recall promedio por clase: validación {info['recall_promedio_val']:.3f} (notebook 02) → test {m_keras['recall_promedio']:.3f}")
if "MEL" in CLASES:
    print(f"Recall de melanoma (MEL) en test: {m_keras['recall_por_clase']['MEL']:.3f}")

ConfusionMatrixDisplay.from_predictions(
    y_test, prob_keras.argmax(axis=1), labels=list(range(K)), display_labels=CLASES,
    normalize="true", cmap="Blues", values_format=".2f",
)
plt.title("Matriz de confusión en test (cada fila suma 1)")
plt.tight_layout()
plt.savefig(TRABAJO / "confusion_test.png", dpi=130)
plt.show()

## 5. Convertir a TFLite

- **float32:** mismos números que Keras. Es el que se usa por defecto.
- **ligero:** cuantización de rango dinámico (pesos en int8, cálculos en float). Unas 4 veces más chico; puede perder un poco de exactitud, por eso se mide aparte.

Si la conversión directa desde Keras falla en esta versión de TensorFlow, se exporta como SavedModel y se convierte desde ahí.

In [ ]:
def convertir(cuantizar):
    def terminar(conv):
        if cuantizar:
            conv.optimizations = [tf.lite.Optimize.DEFAULT]
        return conv.convert()

    try:
        return terminar(tf.lite.TFLiteConverter.from_keras_model(modelo)), "from_keras_model"
    except Exception as e:
        print(f"  from_keras_model no funcionó ({type(e).__name__}); se convierte desde un SavedModel")
        ruta = TRABAJO / "savedmodel"
        shutil.rmtree(ruta, ignore_errors=True)
        modelo.export(str(ruta))
        return terminar(tf.lite.TFLiteConverter.from_saved_model(str(ruta))), "from_saved_model"


tflite = {}
tflite["float32"], via = convertir(cuantizar=False)
print(f"float32: {len(tflite['float32']) / 1e6:.1f} MB (vía {via})")
if EXPORTAR_LIGERO:
    tflite["ligero"], via = convertir(cuantizar=True)
    print(f"ligero:  {len(tflite['ligero']) / 1e6:.1f} MB (vía {via})")

## 6. Medir los `.tflite` con el intérprete de la Pi

Se recorre todo el test, una foto a la vez, igual que hará la Raspberry. El float32 **tiene** que coincidir con Keras; si no, algo se rompió en la conversión y el notebook se detiene.

Los milisegundos por imagen son del CPU de Colab; en la Pi serán distintos.

In [ ]:
def evaluar_tflite(contenido, X):
    it = Interpreter(model_content=contenido, num_threads=HILOS)
    it.allocate_tensors()
    ent, sal = it.get_input_details()[0], it.get_output_details()[0]
    assert ent["dtype"] == np.float32 and tuple(ent["shape"][1:]) == (TAMANO, TAMANO, 3), ent
    prob = np.empty((len(X), K), dtype=np.float32)
    t0 = time.perf_counter()
    for i in range(len(X)):
        it.set_tensor(ent["index"], np.asarray(X[i:i + 1], dtype=np.float32))
        it.invoke()
        prob[i] = it.get_tensor(sal["index"])[0]
    return prob, (time.perf_counter() - t0) / len(X) * 1000


resultados = {"keras": {"mb": None, "ms": ms_keras, **m_keras, "coincide_con_keras": 1.0, "dif_max_prob": 0.0}}
probs = {"keras": prob_keras}
for nombre, contenido in tflite.items():
    prob, ms = evaluar_tflite(contenido, X_test)
    probs[nombre] = prob
    resultados[nombre] = {
        "mb": len(contenido) / 1e6, "ms": ms, **metricas(prob),
        "coincide_con_keras": float((prob.argmax(axis=1) == prob_keras.argmax(axis=1)).mean()),
        "dif_max_prob": float(np.abs(prob - prob_keras).max()),
    }

if resultados["float32"]["dif_max_prob"] > 1e-3:
    raise RuntimeError(f"El TFLite float32 no reproduce a Keras (diferencia máxima {resultados['float32']['dif_max_prob']:.2e}). "
                       "No uses este modelo; revisa la conversión.")

tabla = pd.DataFrame({
    n: {
        "MB": "—" if r["mb"] is None else round(r["mb"], 1),
        "accuracy": round(r["accuracy"], 3),
        "recall promedio": round(r["recall_promedio"], 3),
        "F1 macro": round(r["f1_macro"], 3),
        "recall MEL": round(r["recall_por_clase"].get("MEL", float("nan")), 3),
        "coincide con Keras": f"{100 * r['coincide_con_keras']:.1f} %",
        "ms por imagen (Colab)": round(r["ms"], 1),
    } for n, r in resultados.items()
}).T
display(tabla)

if "ligero" in resultados:
    perdida = resultados["float32"]["recall_promedio"] - resultados["ligero"]["recall_promedio"]
    if perdida <= 0.01:
        print(f"El ligero pierde {100 * perdida:.1f} puntos de recall promedio: puedes usarlo si quieres ahorrar espacio.")
    else:
        print(f"El ligero pierde {100 * perdida:.1f} puntos de recall promedio: usa modelo.tflite (float32).")

## 7. Guardar en Drive y descargar

Además del modelo, se guardan unas fotos del test con las probabilidades que dio aquí (`muestras_pi.npz`). En la sesión 4 la Pi las clasificará y comprobará que obtiene lo mismo.

In [ ]:
import PIL

rng = np.random.default_rng(SEMILLA)
idx_muestras = np.sort(np.concatenate([
    rng.choice(np.flatnonzero(y_test == c), size=min(MUESTRAS_POR_CLASE_PI, int((y_test == c).sum())), replace=False)
    for c in range(K) if (y_test == c).any()
]))
muestras = {"x": np.asarray(X_test[idx_muestras]), "y": y_test[idx_muestras],
            "prob_float32": probs["float32"][idx_muestras]}
if "ligero" in probs:
    muestras["prob_ligero"] = probs["ligero"][idx_muestras]
np.savez_compressed(TRABAJO / "muestras_pi.npz", **muestras)

(TRABAJO / "modelo.tflite").write_bytes(tflite["float32"])
if "ligero" in tflite:
    (TRABAJO / "modelo_ligero.tflite").write_bytes(tflite["ligero"])
(TRABAJO / "clases.txt").write_text("\n".join(CLASES) + "\n")
(TRABAJO / "reporte_test.txt").write_text(reporte)

metricas_json = {
    "fecha": time.strftime("%Y-%m-%d %H:%M"),
    "modo_prueba": MODO_PRUEBA,
    "clases": CLASES,
    "tamano": TAMANO,
    "n_test": int(len(y_test)),
    "preprocesado": meta.get("preprocesado"),
    "entrada": "float32 (1, tamano, tamano, 3) con píxeles 0-255; el escalado va dentro del modelo",
    "entrenamiento": {k: info.get(k) for k in ["fecha", "fase_elegida", "recall_promedio_val", "recall_por_clase_val"]},
    "modelos": {n: {k: v for k, v in r.items()} for n, r in resultados.items()},
    "intérprete": RUNTIME,
    "tensorflow": tf.__version__,
    "keras": keras.__version__,
    "pillow": PIL.__version__,
}
(TRABAJO / "metricas_test.json").write_text(json.dumps(metricas_json, ensure_ascii=False, indent=2))

archivos = ["modelo.tflite"] + (["modelo_ligero.tflite"] if "ligero" in tflite else []) + \
           ["clases.txt", "muestras_pi.npz", "reporte_test.txt", "confusion_test.png", "metricas_test.json"]
(SALIDA / "metricas_test.json").unlink(missing_ok=True)       # se copia al final: marca "terminado"
for nombre in archivos:
    shutil.copy2(TRABAJO / nombre, SALIDA / nombre)

zip_ruta = TRABAJO / "isic_pi_modelo.zip"
with zipfile.ZipFile(zip_ruta, "w", zipfile.ZIP_DEFLATED) as z:
    for nombre in archivos:
        z.write(TRABAJO / nombre, arcname=nombre)
shutil.copy2(zip_ruta, SALIDA / zip_ruta.name)

for p in sorted(SALIDA.iterdir()):
    print(f"{p.name:22s} {p.stat().st_size / 1e6:8.1f} MB")

# Comprobación: el archivo guardado en Drive, leído de cero, reproduce las probabilidades de las muestras
it = Interpreter(model_path=str(SALIDA / "modelo.tflite"))
it.allocate_tensors()
ent, sal = it.get_input_details()[0], it.get_output_details()[0]
for x, esperado in zip(muestras["x"], muestras["prob_float32"]):
    it.set_tensor(ent["index"], x[None].astype(np.float32))
    it.invoke()
    assert np.allclose(it.get_tensor(sal["index"])[0], esperado, atol=1e-5)
print(f"\n✓ modelo.tflite guardado y comprobado con {len(idx_muestras)} fotos de muestra.")

if EN_COLAB:
    from google.colab import files
    files.download(str(zip_ruta))

## Qué quedó y qué sigue

```
Mi unidad/isic-pi/exportado/
├── modelo.tflite          el que usará la Pi (float32)
├── modelo_ligero.tflite   versión con pesos int8 (opcional)
├── clases.txt             una clase por línea; el índice es la posición
├── muestras_pi.npz        fotos de test + probabilidades esperadas, para comprobar en la Pi
├── reporte_test.txt       precisión / recall / F1 por clase en test
├── confusion_test.png
├── metricas_test.json     todas las cifras (se escribe al final)
└── isic_pi_modelo.zip     todo lo anterior junto, para la Mac
```

**En tu Mac:** descomprime `isic_pi_modelo.zip` dentro de la carpeta `modelo/` del repo (si la descarga del navegador falla, bájalo de Drive).

Siguiente sesión: **04 · Script de la Pi** (`pi/clasificar.py`: clasificar fotos y tomar fotos con la cámara).